# 🔧 Module 5.3: Custom PyFunc Models

**Goal**: Build custom MLFlow models with preprocessing, postprocessing, or ensemble logic.

PyFunc models are the most **flexible** way to deploy ML with MLFlow — you can wrap any Python code.

---

In [ ]:
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import json

mlflow.autolog(disable=True)
mlflow.set_experiment("05_custom_pyfunc")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Simple Custom Model — Preprocessing + Model

The most common use case: combine a **scaler** and **model** into a single deployable unit.

In [ ]:
class WineClassifier(mlflow.pyfunc.PythonModel):
    """
    A custom model that includes preprocessing (scaling) 
    and postprocessing (human-readable labels).
    """
    
    def __init__(self, scaler, model, class_names):
        self.scaler = scaler
        self.model = model
        self.class_names = class_names
    
    def predict(self, context, model_input, params=None):
        """
        Make predictions with preprocessing and postprocessing.
        
        Args:
            context: MLFlow context (not used here but required)
            model_input: Input DataFrame
            params: Optional inference params
        
        Returns:
            DataFrame with predicted class names and probabilities
        """
        # 1. Preprocess: scale the input
        scaled_input = self.scaler.transform(model_input)
        
        # 2. Predict
        predictions = self.model.predict(scaled_input)
        probabilities = self.model.predict_proba(scaled_input)
        
        # 3. Postprocess: add human-readable labels and confidence
        results = pd.DataFrame({
            "predicted_class": [self.class_names[p] for p in predictions],
            "confidence": [prob.max() for prob in probabilities],
            "class_id": predictions
        })
        
        return results

# Train the components
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Create the custom model
custom_model = WineClassifier(
    scaler=scaler, 
    model=model,
    class_names=list(wine.target_names)
)

# Test it
result = custom_model.predict(None, X_test[:5])
print("Custom model predictions:")
print(result)
print(f"\n✅ Custom model works!")

In [ ]:
# Log the custom model
with mlflow.start_run(run_name="custom_pyfunc_model"):
    # Infer signature
    signature = infer_signature(X_test[:5], result)
    
    # Log the custom model
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=custom_model,
        signature=signature,
        input_example=X_test[:2],
    )
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test_scaled)))
    
    run_id = mlflow.active_run().info.run_id
    print(f"✅ Custom PyFunc model logged!")
    print(f"   Run ID: {run_id}")

In [ ]:
# Load and use the custom model
loaded = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")

# Note: the input is RAW (unscaled) data — the model handles scaling internally!
predictions = loaded.predict(X_test[:5])
print("Predictions from loaded custom model:")
print(predictions)
print(f"\n🎉 The model handles preprocessing internally — deploy-ready!")

## 2. Ensemble Model — Multiple Models Combined

A more advanced example: combine predictions from multiple models.

In [ ]:
class EnsembleModel(mlflow.pyfunc.PythonModel):
    """Ensemble of multiple sklearn models using majority voting."""
    
    def __init__(self, models, model_names):
        self.models = models
        self.model_names = model_names
    
    def predict(self, context, model_input, params=None):
        # Get predictions from all models
        all_predictions = np.array([
            model.predict(model_input) for model in self.models
        ])
        
        # Majority voting
        from scipy import stats
        ensemble_pred = stats.mode(all_predictions, axis=0, keepdims=False).mode
        
        return ensemble_pred

# Train multiple models
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

# Create ensemble
ensemble = EnsembleModel(
    models=[rf, gb],
    model_names=["RandomForest", "GradientBoosting"]
)

# Test
ensemble_preds = ensemble.predict(None, X_test)
ensemble_acc = accuracy_score(y_test, ensemble_preds)

print(f"Individual accuracies:")
print(f"  RandomForest: {accuracy_score(y_test, rf.predict(X_test)):.4f}")
print(f"  GradientBoosting: {accuracy_score(y_test, gb.predict(X_test)):.4f}")
print(f"  Ensemble: {ensemble_acc:.4f}")

In [ ]:
# Log the ensemble
with mlflow.start_run(run_name="ensemble_pyfunc"):
    signature = infer_signature(X_test, ensemble_preds)
    
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ensemble,
        signature=signature,
        input_example=X_test[:2],
    )
    
    mlflow.log_metric("ensemble_accuracy", ensemble_acc)
    mlflow.set_tag("model_type", "ensemble")
    
    print(f"✅ Ensemble model logged!")

## 🔑 Key Takeaways

1. **Inherit from `mlflow.pyfunc.PythonModel`** and implement `predict()`
2. **`load_context()`** — for loading external artifacts (files, data)
3. **`predict(context, model_input, params)`** — your inference logic
4. Custom models can include **preprocessing**, **postprocessing**, **ensembles**, and any Python logic
5. Once logged, custom models are loaded identically: `mlflow.pyfunc.load_model()`

---

## ➡️ Next: `04_model_evaluation.ipynb` — Systematic model evaluation